# IndoBERT Token Classifier (Colab)

Notebook ini untuk inferensi label token pantun modern dengan skema label BIO (`B/I-ASG`, `B/I-SKT`, `B/I-DSM`, `B/I-TYP`, `O`) memakai model IndoBERT yang disimpan di Google Drive.

Sesuai dokumentasi pipeline:
- `B/I-SKT` -> Rumus A (fonemisasi alfabetis)
- `B/I-DSM` / `B/I-TYP` -> Rumus B & C
- `B/I-ASG` -> failsafe G2P asing
- `O` -> Cascaded CRF 1 (G2P kata baku)


In [1]:
!pip -q install transformers sentencepiece accelerate g2p-id g2p-en nltk

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
# UBAH sesuai lokasi folder model fine-tuned Anda di Google Drive.
# Folder ini minimal berisi: config.json, pytorch_model.bin/model.safetensors,
# tokenizer.json atau vocab.txt + tokenizer config.
MODEL_DIR = '/content/drive/MyDrive/Models/indobert-pantun-ner'  # <- ganti path ini

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForTokenClassification.from_pretrained(MODEL_DIR).to(device)
model.eval()

id2label = model.config.id2label
label2id = model.config.label2id
print('Labels:', id2label)

In [ ]:
def predict_token_labels(text: str):
    """Inferensi token-level label dengan penyelarasan wordpiece -> token asli."""
    words = text.strip().split()
    enc = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors='pt',
        truncation=True,
        padding=False
    )
    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        logits = model(**enc).logits
        pred_ids = torch.argmax(logits, dim=-1)[0].tolist()

    word_ids = tokenizer(words, is_split_into_words=True).word_ids()

    token_labels = []
    current_word = None
    for idx, widx in enumerate(word_ids):
        if widx is None:
            continue
        if widx != current_word:
            current_word = widx
            label = id2label[pred_ids[idx]]
            token_labels.append((words[widx], label))
    return token_labels


LABELS_ASING = {"B-ASG", "I-ASG"}
LABELS_SINGKATAN = {"B-SKT", "I-SKT"}
LABELS_DISEMVOWEL = {"B-DSM", "I-DSM"}
LABELS_TYPO = {"B-TYP", "I-TYP"}
LABELS_VALID = {"O", *LABELS_ASING, *LABELS_SINGKATAN, *LABELS_DISEMVOWEL, *LABELS_TYPO}


def route_by_label(label: str) -> str:
    """Routing sesuai dokumentasi pipeline dan skema label BIO IndoBERT."""
    norm = label.upper().strip()
    if norm not in LABELS_VALID:
        raise ValueError(f'Label tidak dikenali: {label}')

    if norm in LABELS_SINGKATAN:
        return 'RUMUS_A_FONEMISASI_ALFABETIS'
    if norm in LABELS_DISEMVOWEL or norm in LABELS_TYPO:
        return 'RUMUS_B_C_FONEMISASI_DISEMVOWELLING'
    if norm in LABELS_ASING:
        return 'FAILSAFE_G2P_ASING'
    return 'CRF1_G2P_KATA_BAKU'


def is_likely_disemvowelling(token: str) -> bool:
    """Heuristik robust untuk mendeteksi pola disemvowelling tanpa kamus hardcoded."""
    t = token.lower().strip()
    if not t.isalpha() or len(t) < 3:
        return False

    vowels = set('aiueo')
    vowel_count = sum(ch in vowels for ch in t)
    consonant_ratio = (len(t) - vowel_count) / len(t)

    max_cons_run = 0
    curr_run = 0
    for ch in t:
        if ch in vowels:
            curr_run = 0
        else:
            curr_run += 1
            max_cons_run = max(max_cons_run, curr_run)

    consonant_clusters = (
        'ng', 'ny', 'kh', 'sy', 'tr', 'br', 'gr', 'pr', 'kr', 'sk', 'st', 'nd', 'mb'
    )
    cluster_hits = sum(1 for c in consonant_clusters if c in t)

    score = 0
    if vowel_count == 0:
        score += 3
    elif vowel_count == 1 and len(t) >= 4:
        score += 2
    if consonant_ratio >= 0.70:
        score += 2
    if max_cons_run >= 3:
        score += 2
    if cluster_hits >= 1:
        score += 1
    if len(t) >= 4 and t[0] not in vowels and t[-1] not in vowels and vowel_count <= 1:
        score += 1

    return score >= 4


def override_typo_to_dsm(token: str, label: str) -> str:
    norm = label.upper().strip()
    if norm in LABELS_TYPO and is_likely_disemvowelling(token):
        return norm.replace('TYP', 'DSM')
    return norm


def predict_with_routing(text: str):
    labeled = predict_token_labels(text)
    rows = []
    for tok, lab in labeled:
        final_label = override_typo_to_dsm(tok, lab)
        switch = acoustic_switchboard(tok, final_label)
        rows.append(
            {
                'token': tok,
                'raw_label': lab,
                'label': final_label,
                'route': route_by_label(final_label),
                'switch_route': switch['switch_route'],
                'switch_input': switch['switch_input'],
                'phoneme': switch['phoneme'],
                'bc_info': switch.get('bc_info')
            }
        )
    return rows

In [ ]:
# Helper sederhana Rumus A (singkatan)
from functools import lru_cache
import re
import nltk
from g2p_id import G2P as G2P_ID
from g2p_en import G2p as G2P_EN

def _ensure_nltk_resource(resource_path: str, package_name: str):
    try:
        nltk.data.find(resource_path)
    except LookupError:
        nltk.download(package_name, quiet=True)

_ensure_nltk_resource('corpora/cmudict', 'cmudict')
_ensure_nltk_resource('taggers/averaged_perceptron_tagger_eng', 'averaged_perceptron_tagger_eng')
_ensure_nltk_resource('taggers/averaged_perceptron_tagger', 'averaged_perceptron_tagger')
g2p_id_model = G2P_ID()
g2p_en_model = G2P_EN()

ALPHABET_EJA_MAP = {
    "a": "a", "b": "be", "c": "ce", "d": "de", "e": "e", "f": "ef", "g": "ge",
    "h": "ha", "i": "i", "j": "je", "k": "ka", "l": "el", "m": "em", "n": "en",
    "o": "o", "p": "pe", "q": "ki", "r": "er", "s": "es", "t": "te", "u": "u",
    "v": "ve", "w": "we", "x": "eks", "y": "ye", "z": "zet"
}

# Konversi simbol fonem Inggris (ARPAbet) ke fonetik internasional (IPA)
ARPABET_TO_IPA = {
    "AA": "ɑ", "AE": "æ", "AH": "ʌ", "AO": "ɔ", "AW": "aʊ", "AY": "aɪ",
    "B": "b", "CH": "tʃ", "D": "d", "DH": "ð", "EH": "ɛ", "ER": "ɝ",
    "EY": "eɪ", "F": "f", "G": "ɡ", "HH": "h", "IH": "ɪ", "IY": "i",
    "JH": "dʒ", "K": "k", "L": "l", "M": "m", "NG": "ŋ", "OW": "oʊ",
    "OY": "ɔɪ", "P": "p", "R": "r", "S": "s", "SH": "ʃ", "T": "t",
    "TH": "θ", "UH": "ʊ", "UW": "u", "V": "v", "W": "w", "Y": "j",
    "Z": "z", "ZH": "ʒ"
}


def singkatan_to_ejaan(word: str) -> str:
    """dpr -> de pe er"""
    clean = "".join(ch for ch in word.lower() if ch.isalpha())
    return " ".join(ALPHABET_EJA_MAP.get(ch, ch) for ch in clean)


def arpabet_to_ipa(arpabet_tokens: list[str]) -> str:
    ipa_tokens = []
    for token in arpabet_tokens:
        base = re.sub(r"\d", "", token).upper()
        if not base:
            continue
        ipa_tokens.append(ARPABET_TO_IPA.get(base, base.lower()))
    return "".join(ipa_tokens)


def _coerce_g2p_output_to_str(raw) -> str:
    if isinstance(raw, tuple):
        return _coerce_g2p_output_to_str(raw[0])
    if isinstance(raw, list):
        if raw and isinstance(raw[0], list):
            raw = [ph for sub in raw for ph in sub if ph]
        return " ".join(str(part) for part in raw if part)
    return str(raw).strip()


@lru_cache(maxsize=8192)
def g2p_id_phoneme(word: str) -> str:
    if not word:
        return ""
    try:
        res = g2p_id_model.to_phoneme(word)
    except AttributeError:
        try:
            res = g2p_id_model(word)
        except Exception:
            res = g2p_id_model.convert(word)
    return _coerce_g2p_output_to_str(res).replace(" ", "")


@lru_cache(maxsize=8192)
def g2p_foreign_phoneme(word: str) -> str:
    if not word:
        return ""
    raw = g2p_en_model(word)
    phones = [p for p in raw if p and p != " " and p != "<unk>"]
    ipa = arpabet_to_ipa(phones)
    return ipa if ipa else g2p_id_phoneme(word)


def rumus_a_singkatan_simple(word: str) -> str:
    """Rumus A: singkatan -> ejaan alfabet -> fonem Indonesia (g2p-id)."""
    ejaan = singkatan_to_ejaan(word)
    if not ejaan:
        return ""
    return g2p_id_phoneme(ejaan)


# -------------------------------
# Rumus B & C + typo handling (tanpa hardcoded kata)
# -------------------------------

VOWELS = set('aiueo')
COMMON_DIGRAPHS = ("ng", "ny", "kh", "sy", "br", "tr", "kr", "pr", "gr", "dr", "st", "sk")


def normalize_typo_token(token: str) -> str:
    """Normalisasi typo berbasis bentuk: lowercase, alfabet-only, kompres pengulangan huruf."""
    t = token.lower().strip()
    if not t:
        return t
    t = "".join(ch for ch in t if ch.isalpha())
    if not t:
        return t
    # Kompres huruf berulang berlebihan: bangetttt -> bangett
    t = re.sub(r"(.)\1{2,}", r"\1\1", t)
    return t


def _max_consonant_run(t: str) -> int:
    best = 0
    run = 0
    for ch in t:
        if ch in VOWELS:
            run = 0
        else:
            run += 1
            best = max(best, run)
    return best


def _has_common_cluster(t: str) -> bool:
    return any(dg in t for dg in COMMON_DIGRAPHS)


# cek apakah kata mirip dengan disemvowelling (tanpa vokal)
def _is_content_like_disemvowelling(t: str) -> bool:
    vowel_count = sum(ch in VOWELS for ch in t)
    cons_run = _max_consonant_run(t)
    if vowel_count == 0:
        return True
    if vowel_count <= 1 and len(t) >= 4:
        return True
    if cons_run >= 3:
        return True
    if cons_run >= 2 and _has_common_cluster(t):
        return True
    return False


def rumus_c_clamping(word: str, force_two_syllable: bool = False) -> str:
    """Rumus C: clamping panjang kata; opsional dipaksa 2 unit untuk typo pendek."""
    t = word.lower().strip()
    if force_two_syllable:
        return "vone" if t and t[-1] in VOWELS else "vonem"
    if len(t) <= 3:
        return "von"
    return "vone" if t and t[-1] in VOWELS else "vonem"


def rumus_b_phonotactic_mapping(word: str) -> str:
    """Rumus B: estimasi struktur fonotaktik untuk kata konten tanpa vokal."""
    t = "".join(ch for ch in word.lower() if ch.isalpha())
    if not t:
        return "von"

    vowel_count = sum(ch in VOWELS for ch in t)
    cons_run = _max_consonant_run(t)
    has_cluster = _has_common_cluster(t)

    # Tuning untuk kasus disemvowelling umum di pantun gaul:
    # - token tanpa vokal dengan panjang >=5 cenderung 3 suku (skrng, brgkt)
    # - token tanpa vokal dengan panjang <=4 tetap 2 suku (lgsg)
    if vowel_count == 0:
        if len(t) <= 4:
            syll_est = 2
        elif has_cluster or cons_run >= 4:
            syll_est = 3
        else:
            syll_est = 2
        patterns = {2: "vonem", 3: "vonemis", 4: "vonemisa"}
        return patterns[syll_est]

    compact = t
    for dg in COMMON_DIGRAPHS:
        compact = compact.replace(dg, "X")

    base_units = len(compact)
    has_coda = t[-1] in "bcdfghjklmnpqrstvwxyz"
    if has_coda and base_units > 1:
        base_units -= 1

    syll_est = max(2, min(4, (base_units + 1) // 2))

    patterns = {
        2: "vonem",
        3: "vonemis",
        4: "vonemisa",
    }
    return patterns[syll_est]


def rumus_bc_for_dsm_typo(token: str, label: str) -> dict:
    """
    Bedanya:
    - Rumus C: clamping sederhana (panjang token) untuk kata tugas/pendek.
    - Rumus B: mapping fonotaktik untuk kata konten/disemvowelling kompleks.
    """
    norm_label = label.upper().strip()
    normalized_token = normalize_typo_token(token)

    # TYPO pendek (mis. "kta") sering under-count silabel jika diperlakukan 1 unit.
    if norm_label in LABELS_TYPO and len(normalized_token) <= 3:
        return {
            "normalized_token": normalized_token,
            "rule_used": "C",
            "phoneme": rumus_c_clamping(normalized_token, force_two_syllable=True),
        }

    # Token sangat pendek tanpa vokal (mirip "gw") diarahkan ke Rumus C.
    if len(normalized_token) <= 2 and all(ch not in VOWELS for ch in normalized_token):
        return {
            "normalized_token": normalized_token,
            "rule_used": "C",
            "phoneme": rumus_c_clamping(normalized_token),
        }

    # Struktur yang jelas disemvowelling konten -> Rumus B
    if _is_content_like_disemvowelling(normalized_token):
        return {
            "normalized_token": normalized_token,
            "rule_used": "B",
            "phoneme": rumus_b_phonotactic_mapping(normalized_token),
        }

    # Sisanya fallback ke Rumus C.
    if len(normalized_token) <= 3:
        return {
            "normalized_token": normalized_token,
            "rule_used": "C",
            "phoneme": rumus_c_clamping(normalized_token),
        }

    # Lainnya -> Rumus C
    return {
        "normalized_token": normalized_token,
        "rule_used": "C",
        "phoneme": rumus_c_clamping(normalized_token),
    }


def acoustic_switchboard(token: str, label: str) -> dict:
    """Switch board fonem: jalur A, B/C, ASG, atau O sesuai label IndoBERT."""
    norm = label.upper().strip()
    cleaned = normalize_typo_token(token)

    if norm in LABELS_SINGKATAN:
        ejaan = singkatan_to_ejaan(token)
        return {
            "switch_route": "A_SINGKATAN_G2P_ID",
            "switch_input": ejaan,
            "phoneme": rumus_a_singkatan_simple(token),
        }

    if norm in LABELS_ASING:
        return {
            "switch_route": "ASG_G2P_EN_IPA",
            "switch_input": cleaned,
            "phoneme": g2p_foreign_phoneme(cleaned),
        }

    if norm in LABELS_DISEMVOWEL or norm in LABELS_TYPO:
        bc = rumus_bc_for_dsm_typo(token, norm)
        return {
            "switch_route": f"BC_{bc['rule_used']}",
            "switch_input": bc["normalized_token"],
            "phoneme": bc["phoneme"],
            "bc_info": bc,
        }

    return {
        "switch_route": "O_G2P_ID",
        "switch_input": cleaned,
        "phoneme": g2p_id_phoneme(cleaned),
    }


In [ ]:
# Contoh uji
sample_text = 'skrng gw lgsg brgkt ke dpr buat meetup now'
results = predict_with_routing(sample_text)

for row in results:
    bc = row.get('bc_info')
    if bc:
        print(
            f"{row['token']:<12} | raw={row['raw_label']:<8} | final={row['label']:<8} | "
            f"{row['route']:<35} | switch={row['switch_route']:<18} | "
            f"rule={bc['rule_used']} | norm={bc['normalized_token']} | phoneme={row['phoneme']}"
        )
    else:
        print(
            f"{row['token']:<12} | raw={row['raw_label']:<8} | final={row['label']:<8} | "
            f"{row['route']:<35} | switch={row['switch_route']:<18} | phoneme={row['phoneme']}"
        )

## Catatan integrasi tahap berikutnya

Output `route` dari notebook ini bisa langsung dihubungkan ke modul:
1. **Rumus A** untuk singkatan alfabetis.
2. **Rumus B/C** untuk disemvowelling/typo.
3. **Failsafe G2P asing** untuk kata asing.
4. **CRF 1 (G2P kata baku)** untuk label `O`.

Lalu semua deret fonem diarahkan ke **CRF 2 (syllabifier B-I)** untuk kalkulasi metrum/rima.